# m00 · 01 — CRS и расстояния, сделанные правильно

**Что отрабатываем (сессия «CRS and distance, done right»):**
1. Своя реализация haversine.
2. Геодезическое расстояние через `pyproj.Geod` (эллипсоид).
3. Евклидово расстояние в EPSG:3857 (Web Mercator) — и почему оно **врёт**.
4. Евклидово расстояние в локальной UTM-зоне (Алматы — 43N, EPSG:32643).
5. Ошибка Web Mercator на широте города: `1/cos(lat)`.
6. Ловушка `always_xy=True`.

**Результат (dn):** таблица из четырёх расстояний + по два предложения про ошибку каждого.

> **Как работать с этим ноутбуком.** Серые блоки `# >>> ВАШ КОД ЗДЕСЬ` нужно дописать самой и
> удалить `raise NotImplementedError`. После каждого задания стоит `assert` — это проверка,
> которая должна пройти. В папке `solutions/` лежит мой вариант: смотрите в него только после
> того, как попробовали сами (или если застряли больше чем на 20 минут).

In [1]:
import math
import numpy as np
import pandas as pd
from pyproj import Geod, Transformer, CRS

# Две точки в Алматы: площадь Республики и аэропорт
A = (43.2380, 76.9450)   # (lat, lon)
B = (43.3521, 77.0400)
LAT_CITY = 43.25

## Задание 1 — haversine своими руками

Формула (R = 6 371 008.8 м — средний радиус Земли):

$$a = \sin^2\frac{\Delta\varphi}{2} + \cos\varphi_1\cos\varphi_2\sin^2\frac{\Delta\lambda}{2},\qquad d = 2R\,\mathrm{atan2}(\sqrt a, \sqrt{1-a})$$

Функция должна принимать градусы и возвращать **метры**.

In [2]:
R_EARTH = 6_371_008.8

def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres on a sphere."""
    # >>> ВАШ КОД ЗДЕСЬ: переведите градусы в радианы, примените формулу выше
    lat = math.radians(lat2 - lat1)
    lon = math.radians(lon2 - lon1)
    a = math.sin(lat / 2) ** 2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(lon / 2) ** 2
    d = 2 * R_EARTH * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return d

d_hav = haversine_m(*A, *B)
print(f"haversine: {d_hav:,.1f} m")
assert 14_000 < d_hav < 16_000, "ожидаем ~15 км между этими точками"

haversine: 14,835.1 m


## Задание 2 — геодезическое расстояние (эллипсоид WGS84)

`Geod(ellps="WGS84").inv(lon1, lat1, lon2, lat2)` возвращает `(az12, az21, dist)`.
Обратите внимание на порядок **lon, lat** — это первая ловушка pyproj.

In [16]:
geod = Geod(ellps="WGS84")
# >>> ВАШ КОД ЗДЕСЬ: вызовите geod.inv с правильным порядком координат
d_geod = geod.inv(A[1], A[0], B[1], B[0])[2]
print(f"geodesic : {d_geod:,.1f} m")
assert abs(d_geod - d_hav) / d_geod < 0.005, "сфера и эллипсоид расходятся меньше чем на 0.5%"

geodesic : 14,836.5 m


## Задание 3 — евклидово расстояние в EPSG:3857 и в UTM 43N

Создайте два трансформера из EPSG:4326 (**с `always_xy=True`**), спроецируйте точки и посчитайте
обычную евклидову дистанцию `sqrt(dx² + dy²)`.

In [11]:
def euclid(p, q):
    return math.hypot(p[0] - q[0], p[1] - q[1])

# >>> ВАШ КОД ЗДЕСЬ: два Transformer.from_crs(..., always_xy=True); transform(lon, lat)
t_merc = Transformer.from_crs(
    CRS.from_epsg(4326), CRS.from_epsg(3395), always_xy=True
)
t_utm = Transformer.from_crs(
    CRS.from_epsg(4326), CRS.from_epsg(32643), always_xy=True
)

p1_merc = t_merc.transform(A[1], A[0])
p2_merc = t_merc.transform(B[1], B[0])
d_merc = euclid(p1_merc, p2_merc)

p1_utm = t_utm.transform(A[1], A[0])
p2_utm = t_utm.transform(B[1], B[0])
d_utm = euclid(p1_utm, p2_utm)
print(f"mercator : {d_merc:,.1f} m")
print(f"utm 43N  : {d_utm:,.1f} m")

assert abs(d_utm - d_geod) / d_geod < 0.001, "UTM должна совпадать с геодезией в пределах 0.1%"
assert d_merc > d_geod * 1.3, "Mercator на 43° должен завышать длину примерно на 37%"

mercator : 20,352.4 m
utm 43N  : 14,835.3 m


## Задание 4 — объясните разницу числом

Масштабный коэффициент Web Mercator по широте равен $k = 1/\cos\varphi$.
Посчитайте его для Алматы и сравните с фактическим отношением `d_merc / d_geod`.

In [21]:
# >>> ВАШ КОД ЗДЕСЬ: k_theory = 1/cos(lat), k_actual = d_merc / d_geod
lat = math.radians(LAT_CITY)
lon = math.radians(A[1]) 

k_theory = 1/math.cos(lat)
k_actual = d_merc / d_geod
print(f"теория 1/cos(43.25°) = {k_theory:.4f}   факт = {k_actual:.4f}")
assert abs(k_theory - k_actual) < 0.01

table = pd.DataFrame({
    "method": ["haversine (sphere)", "geodesic (WGS84)", "euclid EPSG:3857", "euclid UTM 43N"],
    "metres": [d_hav, d_geod, d_merc, d_utm],
})
table["error_vs_geodesic_%"] = (table.metres / d_geod - 1) * 100
table.round(2)

теория 1/cos(43.25°) = 1.3729   факт = 1.3718


,method,metres,error_vs_geodesic_%
0,haversine (sphere),14835.15,-0.01
1,geodesic (WGS84),14836.49,0.00
2,euclid EPSG:3857,20352.40,37.18
3,euclid UTM 43N,14835.33,-0.01


## Задание 5 — ловушка `always_xy`

Без `always_xy=True` pyproj следует порядку осей **из определения CRS**: у EPSG:4326 это
(lat, lon), у EPSG:32643 — (easting, northing). Запустите ячейку и объясните результат.

In [24]:
t_bad = Transformer.from_crs("EPSG:4326", "EPSG:32643")           # без always_xy
t_good = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)

print("bad  (lon, lat):", t_bad.transform(A[1], A[0]))   # lon подали как lat -> точка улетает
print("bad  (lat, lon):", t_bad.transform(A[0], A[1]))   # так CRS-порядок правильный
print("good (lon, lat):", t_good.transform(A[1], A[0]))

# Вопрос: какой баг предотвращает always_xy=True? Напишите ответ одной строкой:
answer = """
# >>> ВАШ КОД ЗДЕСЬ: допишите этот блок
если использовать always_xy=True, он будет фиксировать порядок (lon, lat), иначе порядок будет по CRS и мб будут ошибки, короче юзаем :D
"""

bad  (lon, lat): (-264121.87913964654, 8752952.41365231)
bad  (lat, lon): (657922.4022865625, 4789081.411424591)
good (lon, lat): (657922.4022865625, 4789081.411424591)
